# 데이터 무결성 검사

1. 데이터셋 점검
labeled 데이터셋 점검 (rule-based, llm-as-a-judge).
- rule based : 형식을 만족하는지
- llm-as-a-judge : 올바른 검색 query가 입력되었는지.슬라이드 데이터를 충분히 가지고 있는지.

In [5]:
import pandas as pd
df_None_original = pd.read_csv('./tool_execution_test_Dataset/test_dataset_None_labeled.csv')
df_Tavily_original = pd.read_csv('./tool_execution_test_Dataset/test_dataset_tavily_labeled.csv')
df_Arxiv_original = pd.read_csv('./tool_execution_test_Dataset/test_dataset_arxiv_labeled.csv')

In [7]:
df = pd.concat([df_None_original, df_Tavily_original, df_Arxiv_original])
df.to_csv('./tool_execution_test_Dataset/test_dataset.csv', index=False, encoding='utf-8-sig')


# Rule-Based

데이터 정확성
- 필수 컬럼 8개(slide_index, title, content, tool_called, tool_name, query, raw_tool_result, tool_node_result)가 모두 존재하는지.
- 예상치 못한 extra 컬럼이 없는지.

컬럼별 정확성
1. slide_index
- null이 아닌 양의 정수값을 가지는지.
2. title
- null이 아닌 string값인 동시에 공백/빈 string도 아닌지.
3. content = null이 아닌 string값인 동시에 공백/빈 string도 아닌지.

4. tool_called:
- True/False 중 1 값을 가지는지.
- string이 아닌 bool 타입으로 제대로 불러와지는지.
5. tool_name
- none,tavily_search,arxiv_tool 중 1 값을 가지는지.

6. query : tool_name이 none이라면 null, 아니라면 null이 아닌 string 값인 동시에 공백/빈 string도 아닌지.
7. raw_tool_result :
- tool_name이 none이라면 null, 아니라면 null이 아닌 string 값인지.
- tool_name이 tavily_search라면 다음과 같은 tool call 형태를 가지고 있는지 구조 검증하기.
    - JSON 파싱 가능한지 (valid JSON).
    - 최상위 필수 키가 존재하는지 (query, follow_up_questions, answer, images, results, response_time, request_id).
    - results가 list 타입이며 비어있지 않은지.
    - results 각 항목에 url, title, content, score 키가 존재하는지.
    - results 각 항목의 각 키에 값이 존재하는지.
    - query 필드 값이 원래 query 컬럼 값과 일치하는지.
예시) {"query": "CATCHSELL 베트남 SNS 판매", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "https://www.slideshare.net/slideshow/sns-vietnamese-influencer-commerce-and-sns-marketing-services-lesell/267014227", "title": "베트남 인플루언서 커머스 및 SNS 마케팅 서비스 르셀( ...", "content": "베트남 국가 대상 수출실적을 100% 확보 가능한 마케팅 서비스 입니다. 단순한 베트남 인플루언서 마케팅이 아닌 판매까지 할 수 있는 결제솔루션을 개발하고 운영중인", "score": 0.5019781, "raw_content": null}, {"url": "https://www.instagram.com/reel/DWioQTGicjm?hl=ko", "title": "베트남 SNS에서 조회수가 높다고 매출이 나오는 것은 ...", "content": "베트남 SNS에서 조회수가 높다고 매출이 나오는 것은 아닙니다. 조회수는 관심이고 매출은 신뢰에서 나옵니다. 그래서 베트남 마케팅에서는 조회수 콘텐츠", "score": 0.33163956, "raw_content": null}, {"url": "https://www.instagram.com/reel/DIxr56spri-", "title": "베트남에선 홈페이지보다 브랜드 계정을 먼저 봅니다. 자사몰 ...", "content": "입니다. 브랜드의 SNS 계정이 홈페이지, 고객센터, 광고창구의 역할을 다 하고 있습니다. 현지화 마케팅, 계정 관리부터 시작하세요. #베트남마케팅 #SNS", "score": 0.30093092, "raw_content": null}], "response_time": 1.21, "request_id": "cea5dc90-f5d8-4c53-97ce-87fd8717f4b8"}


- tool_name이 arxiv_tool이라면 '제목', '연도', '요약', 'URL'을 가지는 string 형태인지 구조 검증.
    - 제목:, 연도:, 요약:, URL: 키워드가 모두 포함되어 있는지
    - 연도 : 값이 4자리 숫자인지 (예 : 2024)
제목: Morescient GAI for Software Engineering (Extended Version)
연도: 2024
요약: The ability of Generative AI (GAI) technology to automatically check, synthesize and modify software engineering artifacts promises to revolutionize all aspects of software engineering. Using GAI for software engineering tasks is consequently one of the most rapidly expanding fields of software engineering research, with over a hundred LLM-based code models having been published since 2021. However, the overwhelming majority of existing code models share a major weakness - they are exclusively trained on the syntactic facet of software, significantly lowering their trustworthiness in tasks dependent on software semantics. To address this problem, a new class of "Morescient" GAI is needed that is "aware" of (i.e., trained on) both the semantic and static facets of software. This, in turn, will require a new generation of software observation platforms capable of generating large quantities of execution observations in a structured and readily analyzable way. In this paper, we present a vision and roadmap for how such "Morescient" GAI models can be engineered, evolved and disseminated according to the principles of open science.
URL: http://arxiv.org/abs/2406.04710v2

8. tool_node_result : null이 아닌 string값. 동시에 공백/빈 string도 아닌지.

9. 교차 필드 일관성
tool_called & tool_name 일관성
- tool_called=True → tool_name이 none이 아니어야 함
- tool_called=False → tool_name이 정확히 none이어야 함
tool_called & query 일관성
- tool_called=False → query=null
- tool_called=True → query가 not null
tool_called & raw_tool_result 일관성
- tool_called=False → raw_tool_result=null
- tool_called=True → raw_tool_result not null

In [8]:
df_original = pd.read_csv('./tool_execution_test_Dataset/test_dataset.csv')
df = df_original.copy()
print(f"로드 완료: {len(df)}행  |  컬럼: {list(df.columns)}")
df.head(2)


로드 완료: 131행  |  컬럼: ['slide_index', 'title', 'content', 'tool_called', 'tool_name', 'query', 'raw_tool_result', 'tool_node_result']


,slide_index,title,content,tool_called,tool_name,query,raw_tool_result,tool_node_result
0,1,제목 없음,"- 제목: 제목 없음\n- 본문 텍스트: """", """", """", """", """"\n- 표...",False,none,NaN,NaN,"검색 필요 없음: 표지 슬라이드로 판단됩니다. 구체적인 제품/서비스 기능, 성능, ..."
1,2,제목 없음,- 제목: 제목 없음\n\n- 텍스트(추출 순서 그대로)\n 1) [공란]\n ...,False,none,NaN,NaN,"검색 필요 없음: 제품 차별성, 사내 On-premise LLM 필요성, On-pr..."


In [14]:
import json
import re

REQUIRED_COLS = {'slide_index', 'title', 'content', 'tool_called', 'tool_name',
                 'query', 'raw_tool_result', 'tool_node_result'}
VALID_TOOL_NAMES = {'none', 'tavily_search', 'arxiv_tool'}
TAVILY_TOP_KEYS = {'query', 'follow_up_questions', 'answer', 'images', 'results', 'response_time', 'request_id'}
RESULT_KEYS = {'url', 'title', 'content', 'score'}
ARXIV_KEYWORDS = ['제목:', '연도:', '요약:', 'URL:']
YEAR_RE = re.compile(r'연도:\s*(\d{4})')

errors = []

def report(check_name, failed_pairs):
    """failed_pairs: list of (row_index, slide_index_value) tuples"""
    if len(failed_pairs) == 0:
        print(f"[PASS] {check_name}")
    else:
        sids = [p[1] for p in failed_pairs]
        print(f"[FAIL] {check_name} — {len(failed_pairs)}건 오류")
        print(f"       slide_index: {sids}")
    errors.append({
        "check": check_name,
        "failed_indices": [p[0] for p in failed_pairs],
        "failed_sids": [p[1] for p in failed_pairs],
        "count": len(failed_pairs)
    })

def sid(row, idx):
    val = row.get('slide_index') if hasattr(row, 'get') else None
    if val is not None and not (isinstance(val, float) and val != val):
        return val
    return f"row#{idx}"


# ── Check 0: 컬럼 정확성 ────────────────────────────────────────────────────
actual_cols = set(df.columns)
missing = REQUIRED_COLS - actual_cols
extra = actual_cols - REQUIRED_COLS
if not missing and not extra:
    print("[PASS] Check 0: 컬럼 정확성")
else:
    print("[FAIL] Check 0: 컬럼 정확성")
    if missing: print(f"       누락 컬럼: {missing}")
    if extra:   print(f"       불필요 컬럼: {extra}")
errors.append({"check": "Check 0: 컬럼 정확성",
               "failed_indices": list(missing | extra),
               "failed_sids": list(missing | extra),
               "count": len(missing | extra)})


# ── Check 1: slide_index — not null, 양의 정수 ──────────────────────────────
failed = []
for i, row in df.iterrows():
    v = row['slide_index']
    if pd.isna(v) or not isinstance(v, (int, float)) or int(v) != v or int(v) <= 0:
        failed.append((i, sid(row, i)))
report("Check 1: slide_index (not null, 양의 정수)", failed)


# ── Check 2: title — not null, not blank ────────────────────────────────────
failed = []
for i, row in df.iterrows():
    v = row['title']
    if pd.isna(v) or not isinstance(v, str) or v.strip() == "":
        failed.append((i, sid(row, i)))
report("Check 2: title (not null, not blank)", failed)


# ── Check 3: content — not null, not blank ──────────────────────────────────
failed = []
for i, row in df.iterrows():
    v = row['content']
    if pd.isna(v) or not isinstance(v, str) or v.strip() == "":
        failed.append((i, sid(row, i)))
report("Check 3: content (not null, not blank)", failed)


# ── Check 4: tool_called — bool 타입, True/False ────────────────────────────
failed = []
for i, row in df.iterrows():
    if not isinstance(row['tool_called'], bool):
        failed.append((i, sid(row, i)))
report("Check 4: tool_called (bool 타입, True/False)", failed)


# ── Check 5: tool_name — 허용값 3가지 ──────────────────────────────────────
failed = []
for i, row in df.iterrows():
    v = row['tool_name']
    if pd.isna(v) or v not in VALID_TOOL_NAMES:
        failed.append((i, sid(row, i)))
report("Check 5: tool_name (none/tavily_search/arxiv_tool)", failed)


# ── Check 6: query — tool_name 조건부 ──────────────────────────────────────
failed_none, failed_tool = [], []
for i, row in df.iterrows():
    tn, q = row['tool_name'], row['query']
    if tn == 'none':
        if not pd.isna(q):
            failed_none.append((i, sid(row, i)))
    else:
        if pd.isna(q) or not isinstance(q, str) or q.strip() == "":
            failed_tool.append((i, sid(row, i)))
report("Check 6a: query (tool_name=none → null)", failed_none)
report("Check 6b: query (tool_name≠none → not null, not blank)", failed_tool)


# ── Check 7a: raw_tool_result — tool_name=none → null ──────────────────────
failed = []
for i, row in df.iterrows():
    if row['tool_name'] == 'none' and not pd.isna(row['raw_tool_result']):
        failed.append((i, sid(row, i)))
report("Check 7a: raw_tool_result (tool_name=none → null)", failed)

# ── Check 7b: raw_tool_result — tool_name≠none → not null ──────────────────
failed = []
for i, row in df.iterrows():
    if row['tool_name'] != 'none' and pd.isna(row['raw_tool_result']):
        failed.append((i, sid(row, i)))
report("Check 7b: raw_tool_result (tool_name≠none → not null)", failed)

# ── Check 7c: tavily_search 구조 검증 ───────────────────────────────────────
f_json, f_keys, f_res_type, f_res_item_keys, f_res_item_vals, f_query_match = [], [], [], [], [], []
for i, row in df.iterrows():
    if row['tool_name'] != 'tavily_search':
        continue
    raw = row['raw_tool_result']
    try:
        parsed = json.loads(raw)
    except Exception:
        f_json.append((i, sid(row, i)))
        continue
    if not TAVILY_TOP_KEYS.issubset(parsed.keys()):
        f_keys.append((i, sid(row, i)))
        continue
    res_list = parsed.get('results')
    if not isinstance(res_list, list) or len(res_list) == 0:
        f_res_type.append((i, sid(row, i)))
        continue
    item_key_fail = item_val_fail = False
    for item in res_list:
        if not RESULT_KEYS.issubset(item.keys()):
            item_key_fail = True
            break
        for k in RESULT_KEYS:
            v = item[k]
            if v is None or (isinstance(v, str) and v.strip() == ""):
                item_val_fail = True
                break
    if item_key_fail:
        f_res_item_keys.append((i, sid(row, i)))
        continue
    if item_val_fail:
        f_res_item_vals.append((i, sid(row, i)))
    if parsed.get('query') != row['query']:
        f_query_match.append((i, sid(row, i)))

report("Check 7c-1: tavily JSON 파싱 가능", f_json)
report("Check 7c-2: tavily 최상위 필수 키 존재", f_keys)
report("Check 7c-3: tavily results 리스트 & 비어있지 않음", f_res_type)
report("Check 7c-4: tavily results 각 항목 필수 키 (url,title,content,score)", f_res_item_keys)
report("Check 7c-5: tavily results 각 항목 키 값 존재", f_res_item_vals)
report("Check 7c-6: tavily raw_tool_result.query == query 컬럼 값 일치", f_query_match)

# ── Check 7d: arxiv_tool 구조 검증 ──────────────────────────────────────────
f_kw, f_year = [], []
for i, row in df.iterrows():
    if row['tool_name'] != 'arxiv_tool':
        continue
    raw = row['raw_tool_result']
    if not isinstance(raw, str):
        continue
    if not all(kw in raw for kw in ARXIV_KEYWORDS):
        f_kw.append((i, sid(row, i)))
        continue
    if not YEAR_RE.search(raw):
        f_year.append((i, sid(row, i)))

report("Check 7d-1: arxiv 필수 키워드 (제목:,연도:,요약:,URL:)", f_kw)
report("Check 7d-2: arxiv 연도 4자리 숫자", f_year)


# ── Check 8: tool_node_result — not null, not blank ─────────────────────────
failed = []
for i, row in df.iterrows():
    v = row['tool_node_result']
    if pd.isna(v) or not isinstance(v, str) or v.strip() == "":
        failed.append((i, sid(row, i)))
report("Check 8: tool_node_result (not null, not blank)", failed)


# ── Check 9: 교차 필드 일관성 ────────────────────────────────────────────────
f_t2n, f_f2n = [], []  # tool_called ↔ tool_name
f_t2q, f_f2q = [], []  # tool_called ↔ query
f_t2r, f_f2r = [], []  # tool_called ↔ raw_tool_result

for i, row in df.iterrows():
    tc, tn = row['tool_called'], row['tool_name']
    q, rtr = row['query'], row['raw_tool_result']
    if not isinstance(tc, bool):
        continue
    p = (i, sid(row, i))
    if tc and tn == 'none':          f_t2n.append(p)
    if not tc and tn != 'none':      f_f2n.append(p)
    if tc and pd.isna(q):            f_t2q.append(p)
    if not tc and not pd.isna(q):    f_f2q.append(p)
    if tc and pd.isna(rtr):          f_t2r.append(p)
    if not tc and not pd.isna(rtr):  f_f2r.append(p)

report("Check 9a: tool_called=True  → tool_name ≠ none", f_t2n)
report("Check 9b: tool_called=False → tool_name = none", f_f2n)
report("Check 9c: tool_called=True  → query not null", f_t2q)
report("Check 9d: tool_called=False → query = null", f_f2q)
report("Check 9e: tool_called=True  → raw_tool_result not null", f_t2r)
report("Check 9f: tool_called=False → raw_tool_result = null", f_f2r)


[PASS] Check 0: 컬럼 정확성
[PASS] Check 1: slide_index (not null, 양의 정수)
[PASS] Check 2: title (not null, not blank)
[PASS] Check 3: content (not null, not blank)
[PASS] Check 4: tool_called (bool 타입, True/False)
[PASS] Check 5: tool_name (none/tavily_search/arxiv_tool)
[PASS] Check 6a: query (tool_name=none → null)
[PASS] Check 6b: query (tool_name≠none → not null, not blank)
[PASS] Check 7a: raw_tool_result (tool_name=none → null)
[PASS] Check 7b: raw_tool_result (tool_name≠none → not null)
[PASS] Check 7c-1: tavily JSON 파싱 가능
[PASS] Check 7c-2: tavily 최상위 필수 키 존재
[PASS] Check 7c-3: tavily results 리스트 & 비어있지 않음
[PASS] Check 7c-4: tavily results 각 항목 필수 키 (url,title,content,score)
[PASS] Check 7c-5: tavily results 각 항목 키 값 존재
[PASS] Check 7c-6: tavily raw_tool_result.query == query 컬럼 값 일치
[FAIL] Check 7d-1: arxiv 필수 키워드 (제목:,연도:,요약:,URL:) — 1건 오류
       slide_index: [32]
[PASS] Check 7d-2: arxiv 연도 4자리 숫자
[PASS] Check 8: tool_node_result (not null, not blank)
[PASS] Check 9a: tool_called

In [15]:
print("=" * 60)
print("최종 결과 요약")
print("=" * 60)
total = len(errors)
pass_count = sum(1 for e in errors if e["count"] == 0)
fail_count = total - pass_count
print(f"총 {total}개 검사 | PASS: {pass_count} | FAIL: {fail_count}")
if fail_count > 0:
    print("\n실패한 검사:")
    for e in errors:
        if e["count"] > 0:
            print(f"  - {e['check']}: {e['count']}건 → index {e['failed_indices']}")
else:
    print("\n모든 검사 통과!")


최종 결과 요약
총 25개 검사 | PASS: 24 | FAIL: 1

실패한 검사:
  - Check 7d-1: arxiv 필수 키워드 (제목:,연도:,요약:,URL:): 1건 → index [111]


정제

In [16]:
df.iloc[111] #arxiv Error 때문에 삭제.

slide_index                                                        32
title                                                           제목 없음
content             Knowledge Intelligence Lab. Hanbat National Un...
tool_called                                                      True
tool_name                                                  arxiv_tool
query               Iter-RetGen iterative retrieval generation syn...
raw_tool_result     Error: arxiv.HTTPError('https://export.arxiv.o...
tool_node_result    - 검색 대상 핵심 주제: **Enhancing Retrieval-Augmented...
Name: 111, dtype: object

In [23]:
df = df.drop(111, axis=0).reset_index(drop=True)
df.to_csv('./tool_execution_test_Dataset/test_dataset_cleaned1.csv', index=False, encoding='utf-8-sig')

# LLM-as-a-Judge

In [49]:
import json
import os
from dotenv import dotenv_values
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

# .env 파일을 절대경로로 직접 읽고 key strip 처리
_env_path = os.path.join(os.path.dirname(os.path.abspath('.')), 
                         'PycharmProjects', 'Educational_Video_Generator', '.env')
config = dotenv_values('./.env')
api_key = config.get('OPENAI_API_KEY', '').strip()
print(f"Key 로드 확인: {api_key[:15]}...{api_key[-4:]}  (길이: {len(api_key)})")

df_judge_input = pd.read_csv('tool_execution_test_Dataset/test_dataset_cleaned1.csv')
print(f"평가 대상: {len(df_judge_input)}행")

llm = ChatOpenAI(model="gpt-5", temperature=0, openai_api_key=api_key)


Key 로드 확인: sk-proj-fk0_o2Z...MUsA  (길이: 164)
평가 대상: 130행


In [54]:
JUDGE_SYSTEM_PROMPT = """당신은 검색 품질 평가 전문가입니다.
아래는 PPT 슬라이드를 보고 도구를 선택하는 검색 에이전트의 기준입니다. 이 기준을 완전히 숙지한 뒤, 에이전트의 판단이 올바른지 평가하세요.

---
# 도구 선택 기준

1. tavily_search : 특정 제품/서비스에 관련한 내용을 담은 슬라이드일 때 사용. 해당 제품/서비스에 대한 최신 정보와 동향을 검색합니다.

tavily 검색 필요 예시:
- MAAL (Multilingual Adaptive Augmentation Language-model) 한국어에 강한 언어 생성 모델 MAAL을 기반으로 On-premise LLM 솔루션을 제공합니다. 다양한 파라미터 수의 모델(8B-70B)을 기반으로 고객의 니즈에 맞는 모델을 지원합니다. On-premise용으로는 성능이 뛰어나며 범용적으로 사용하기 좋은 MAAL-albatross(70B)를 권장하고 있습니다. → MAAL 관련 검색
- Agados UI, Flow Design & Visibility Technologies Structure of this presentation Application을 위한 Architecture - SW Package를 위한 Smart Architecture - Hybrid Architecture Overview - 타 시스템과의 Interface → Agados 관련 검색

2. arxiv_tool : 연구, 성능 그래프, 성능 표, 논문 인용 표기 등을 담은 슬라이드일 때 사용. 해당 내용과 관련한 논문을 검색합니다.

arxiv 검색 필요 예시:
- "REPLUG: Retrieval-Augmented Black-Box Language Models, NAACL24'", "Dense Passage Retrieval for Open-Domain Question Answering, EMNLP20'" → 논문 검색
- 마크다운 형식으로 전환된 그래프의 성능, 메트릭 표 → 논문 검색
- 특정 모델의 벤치마크 점수가 수치로 제시된 경우 예) "MAAL 70B: 9.06 / GPT-4o: 9.59" 처럼 모델별 점수 비교표가 포함된 슬라이드 → 논문 검색
- LogicKor, KoBEST, MMLU 등 평가 지표명이 명시된 경우 → 논문 검색

3. none (검색 불필요): 다음 경우에 해당하면 검색하지 않습니다.
- 표지, 개요, 목차, 섹션 구분, 마지막 페이지(Q&A, 감사합니다) 등의 슬라이드
- 소개: 학습 목표, 강사 소개, 참고문헌 목록, 레퍼런스 목록 등을 소개하는 내용
- 그 외 위의 도구 선택 기준에 해당하지 않는 경우

검색 불필요 예시:
- 제목: 제목 없음 - Chapter 2. 오픈소스 컨설팅의 On-premise LLM 솔루션 - MAAL 기반 On-premise 패키지 - 챗봇 - Chatplay → 목차 슬라이드
- Biz. Application을 위한 디자이너/재조정기 '아가도스'는 귀사의 SW Application내에서 Configure Tool의 역할 수행 → 섹션 구분 슬라이드
- 제목: Jamcracker 소개 시작 Cloud Management Platform & Cloud Service Brokerage → 목차 슬라이드
---

# 평가 항목

1. tool_called_correct (true/false)
   content를 보고 검색이 필요한 슬라이드인지 판단하여 에이전트의 tool_called 값이 올바른지 평가합니다.

2. tool_name_correct (true/false)
   - tool_called=false인데 tool_name이 none이 아니면 → false
   - tool_called=true인데 잘못된 도구를 선택했으면 → false (예: 논문 내용인데 tavily_search 선택)
   - tool_called와 tool_name이 모두 올바르면 → true

3. query_correct (true/false/null)
   - tool_name이 none이면 → null (평가 대상 아님)
   - tool_name이 tavily_search: 핵심 제품/서비스 키워드를 포함하고 3~5단어로 간결한지 평가
   - tool_name이 arxiv_tool: 핵심 연구/논문 주제 키워드를 포함하고 단일 주제에 집중하는지 평가

# 출력 형식
반드시 아래 JSON만 출력하세요. 다른 텍스트는 절대 출력하지 마세요.
{"tool_called_correct": true 또는 false, "tool_name_correct": true 또는 false, "query_correct": true 또는 false 또는 null, "reason": "한 문장으로 판단 이유"}"""


def evaluate_row(row):
    query_val = row['query'] if pd.notna(row['query']) else 'null'
    user_content = f"""슬라이드 내용(content):
{row['content']}

에이전트의 판단:
- tool_called: {row['tool_called']}
- tool_name: {row['tool_name']}
- query: {query_val}"""

    try:
        response = llm.invoke([
            SystemMessage(content=JUDGE_SYSTEM_PROMPT),
            HumanMessage(content=user_content)
        ])
        text = response.content.strip()
        if text.startswith("```"):
            text = text.split("```")[1]
            if text.startswith("json"):
                text = text[4:]
        return json.loads(text.strip())
    except json.JSONDecodeError:
        return {
            "tool_called_correct": None,
            "tool_name_correct": None,
            "query_correct": None,
            "reason": f"JSON 파싱 실패: {response.content[:200]}"
        }
    except Exception as e:
        return {
            "tool_called_correct": None,
            "tool_name_correct": None,
            "query_correct": None,
            "reason": f"오류: {str(e)}"
        }


In [51]:
import time

judge_results = []
total = len(df_judge_input)

for i, (idx, row) in enumerate(df_judge_input.iterrows()):
    result = evaluate_row(row)
    judge_results.append(result)

    tc = result['tool_called_correct']
    tn = result['tool_name_correct']
    qc = result['query_correct']
    print(f"[{i+1:3d}/{total}] row={idx:3d} | tool_called={tc} | tool_name={tn} | query={qc}")

# judge 결과를 원본 df에 새 칼럼으로 추가
df_judge_results = pd.DataFrame(judge_results)

df_result = df_judge_input.copy()
df_result['tool_called_correct'] = df_judge_results['tool_called_correct'].values
df_result['tool_name_correct']   = df_judge_results['tool_name_correct'].values
df_result['query_correct']       = df_judge_results['query_correct'].values
df_result['judge_reason']        = df_judge_results['reason'].values

df_result.to_csv('./tool_execution_test_Dataset/test_dataset_cleaned2.csv', index=False, encoding='utf-8-sig')
print(f"\n저장 완료: ./tool_execution_test_Dataset/test_dataset_cleaned2.csv")
print(df_result[['slide_index', 'tool_called_correct', 'tool_name_correct', 'query_correct', 'judge_reason']].to_string())


[  1/130] row=  0 | tool_called=True | tool_name=True | query=None
[  2/130] row=  1 | tool_called=True | tool_name=True | query=None
[  3/130] row=  2 | tool_called=True | tool_name=True | query=None
[  4/130] row=  3 | tool_called=True | tool_name=True | query=None
[  5/130] row=  4 | tool_called=True | tool_name=True | query=None
[  6/130] row=  5 | tool_called=True | tool_name=True | query=None
[  7/130] row=  6 | tool_called=True | tool_name=True | query=None
[  8/130] row=  7 | tool_called=True | tool_name=True | query=None
[  9/130] row=  8 | tool_called=True | tool_name=True | query=None
[ 10/130] row=  9 | tool_called=True | tool_name=True | query=None
[ 11/130] row= 10 | tool_called=True | tool_name=True | query=None
[ 12/130] row= 11 | tool_called=True | tool_name=True | query=None
[ 13/130] row= 12 | tool_called=False | tool_name=False | query=None
[ 14/130] row= 13 | tool_called=True | tool_name=True | query=None
[ 15/130] row= 14 | tool_called=True | tool_name=True | quer

In [52]:
print("=" * 60)
print("LLM-as-a-Judge 결과 요약")
print("=" * 60)

total = len(df_result)
tc_pass = int(df_result['tool_called_correct'].sum())
tn_pass = int(df_result['tool_name_correct'].sum())
q_rows  = df_result[df_result['query_correct'].notna()]
q_pass  = int(q_rows['query_correct'].sum())

print(f"tool_called 정확도: {tc_pass}/{total} ({tc_pass/total:.1%})")
print(f"tool_name 정확도:   {tn_pass}/{total} ({tn_pass/total:.1%})")
print(f"query 적절성:       {q_pass}/{len(q_rows)} ({q_pass/len(q_rows):.1%})  ← 검색 있는 행만")

for label, col in [("tool_called 오류", "tool_called_correct"),
                   ("tool_name 오류",   "tool_name_correct"),
                   ("query 오류",       "query_correct")]:
    fail_rows = df_result[df_result[col] == False]
    if len(fail_rows) > 0:
        print(f"\n--- {label} ({len(fail_rows)}건) ---")
        for idx, r in fail_rows.iterrows():
            print(f"  row={idx}  slide_index={r['slide_index']}: {r['judge_reason']}")


LLM-as-a-Judge 결과 요약
tool_called 정확도: 114/130 (87.7%)
tool_name 정확도:   114/130 (87.7%)
query 적절성:       86/88 (97.7%)  ← 검색 있는 행만

--- tool_called 오류 (12건) ---
  row=12  slide_index=13: 슬라이드 내용이 연구와 관련된 주제를 다루고 있어 arxiv_tool이 적합합니다.
  row=16  slide_index=17: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=19  slide_index=20: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=20  slide_index=21: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=60  slide_index=31: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=66  slide_index=37: 슬라이드는 데이터 출처 설명으로 검색이 필요하지 않습니다.
  row=67  slide_index=38: 슬라이드는 특정 제품/서비스에 대한 내용이 아니므로 검색이 필요하지 않습니다.
  row=68  slide_index=39: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=70  slide_index=41: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=74  slide_index=45: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=77  slide_index=48: 슬라이드는 특정 제품/서비스에 대한 정보가 아닌 일반적인 마케팅 전략에 관한 내용으로 검색이 필요하지 않습니다.
  row=124  slide_index=46: 슬라이드에 연구나 논문 관련 내용이 없어 검색이 필요하지 않습니다.


============================================================ <br>
LLM-as-a-Judge 결과 요약<br>
============================================================<br>
tool_called 정확도: 114/130 (87.7%)<br>
tool_name 정확도:   114/130 (87.7%)<br>
query 적절성:       86/88 (97.7%)  ← 검색 있는 행만<br>

--- tool_called 오류 (12건) ---
  row=12  slide_index=13: 슬라이드 내용이 연구와 관련된 주제를 다루고 있어 arxiv_tool이 적합합니다.
  row=16  slide_index=17: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=19  slide_index=20: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=20  slide_index=21: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=60  slide_index=31: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=66  slide_index=37: 슬라이드는 데이터 출처 설명으로 검색이 필요하지 않습니다.
  row=67  slide_index=38: 슬라이드는 특정 제품/서비스에 대한 내용이 아니므로 검색이 필요하지 않습니다.
  row=68  slide_index=39: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=70  slide_index=41: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=74  slide_index=45: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=77  slide_index=48: 슬라이드는 특정 제품/서비스에 대한 정보가 아닌 일반적인 마케팅 전략에 관한 내용으로 검색이 필요하지 않습니다.
  row=124  slide_index=46: 슬라이드에 연구나 논문 관련 내용이 없어 검색이 필요하지 않습니다.

--- tool_name 오류 (12건) ---
  row=12  slide_index=13: 슬라이드 내용이 연구와 관련된 주제를 다루고 있어 arxiv_tool이 적합합니다.
  row=16  slide_index=17: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=19  slide_index=20: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=20  slide_index=21: Agados 관련 내용이므로 tavily_search가 필요합니다.
  row=60  slide_index=31: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=66  slide_index=37: 슬라이드는 데이터 출처 설명으로 검색이 필요하지 않습니다.
  row=67  slide_index=38: 슬라이드는 특정 제품/서비스에 대한 내용이 아니므로 검색이 필요하지 않습니다.
  row=68  slide_index=39: 슬라이드는 특정 제품/서비스나 논문 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=70  slide_index=41: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=74  slide_index=45: 슬라이드는 특정 제품/서비스나 연구 관련 내용이 아니므로 검색이 필요하지 않습니다.
  row=77  slide_index=48: 슬라이드는 특정 제품/서비스에 대한 정보가 아닌 일반적인 마케팅 전략에 관한 내용으로 검색이 필요하지 않습니다.
  row=124  slide_index=46: 슬라이드에 연구나 논문 관련 내용이 없어 검색이 필요하지 않습니다.

--- query 오류 (2건) ---
  row=82  slide_index=3: 슬라이드 내용은 연구 결과와 성능 차이를 다루고 있어 arxiv_tool이 적절하지만, 쿼리가 너무 길고 여러 주제를 포함하고 있어 부적절합니다.
  row=98  slide_index=19: query가 너무 길고 복잡하며 핵심 연구 주제에 집중하지 못함

In [58]:
judge_cols = ['tool_called_correct', 'tool_name_correct', 'query_correct']

mask = df_result[judge_cols].apply(lambda col: col == False).any(axis=1)
df_failed = df_result[mask].reset_index(names='row_index')

print(f"False 값이 하나 이상 있는 행: {len(df_failed)}건\n")

for _, r in df_failed.iterrows():
    print("━" * 80)
    print(f"  row={int(r['row_index'])}  |  slide_index={r['slide_index']}  |  tool_name={r['tool_name']}")
    print("━" * 80)
    print("[content]")
    print(r['content'])
    print()
    print("[평가 결과]")
    print(f"  tool_called_correct : {r['tool_called_correct']}")
    print(f"  tool_name_correct   : {r['tool_name_correct']}")
    print(f"  query_correct       : {r['query_correct']}")
    print()
    print("[judge_reason]")
    print(f"  {r['judge_reason']}")
    print()


False 값이 하나 이상 있는 행: 14건

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  row=12  |  slide_index=13  |  tool_name=none
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[content]
Chapter 1

Knowledge Intelligence Lab. Hanbat National University

Retrievals
1) Sparse Retrieval
2) Benchmark Datasets
3) Dense Retrieval
4) Negative Sampling
5) Single Vector Retriever
6) Multi Vector Retriever
7) Generative Retrieval

13
Chapter 1

[평가 결과]
  tool_called_correct : False
  tool_name_correct   : False
  query_correct       : None

[judge_reason]
  슬라이드 내용이 연구와 관련된 주제를 다루고 있어 arxiv_tool이 적합합니다.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  row=16  |  slide_index=17  |  tool_name=none
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[content]
- 제목: 1. Agados 기능과 특징

- 텍스트:
  - 1. Agados 기능과 특징
  - Agados UI , Flow Design & Visibility Technologies
  - Functions &

In [ ]:
[유지할 슬라이드]
12 19 20 : 목차 부분이 맞으므로 제외
60 : PXF External Table 관련 내용이 맞으므로 제외
66, 67, 68 : Statista에 관한 내용이 맞으므로 제외
74, 77 : 카카오싱크에 관한 내용이 맞으므로 제외
82, 98 : 연구 결과 자체가 길어 query도 길어짐

[삭제할 슬라이드]
70 : VietSurvey라는 제품이 PPT에 명확히 명시되어 있지 않으므로 삭제
124 : 연구 성과라고 보기엔 일반적인 base model과 small base model의 차이점에 대한 설명이므로 삭제

In [60]:
df_cleaned3 = df_result.drop(index=[70, 124]).reset_index(drop=True)
df_cleaned3.to_csv('./tool_execution_test_Dataset/test_dataset_cleaned3.csv', index=False, encoding='utf-8-sig')
print(f"저장 완료: {len(df_result)}행 → {len(df_cleaned3)}행")
print(f"제거된 행: index 71 (slide_index={df_result.loc[71, 'slide_index']}), index 125 (slide_index={df_result.loc[125, 'slide_index']})")

저장 완료: 130행 → 128행
제거된 행: index 71 (slide_index=42), index 125 (slide_index=47)


최종 데이터셋 : test_dataset_cleaned

In [61]:
df_cleaned3.to_csv('./tool_execution_test_Dataset/test_dataset_cleaned.csv', index=False, encoding='utf-8-sig')